# Application-Specific Tools: Practice Exercise - SOLUTION

Build a production-grade book information tool for a LangChain agent using the **Open Library API** - a real, free API with millions of books.

**What you'll implement:**
- An API helper function that fetches data from the Open Library API
- A Pydantic model for book query validation
- A book search tool with proper error handling and logging

**Why Open Library?**
- Completely free, no API key required
- Real-world data (millions of books from libraries worldwide)
- Production-realistic API patterns (rate limits, network errors, data validation)

**Estimated time:** 20-25 minutes

## Setup

Run this cell to import all required libraries and configure the environment.

In [12]:
# Setup - run this cell first

import os
import logging
import time
from typing import Optional

import requests
from dotenv import load_dotenv
from pydantic import BaseModel, Field, field_validator

from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent

# Load environment variables
load_dotenv()

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY not found in environment variables")

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)
logger = logging.getLogger('book_tools')

print("Setup complete!")

Setup complete!


## Part 1: Implement the API Helper Function (SOLUTION)

The Open Library API is a free, public API that provides access to book data from libraries worldwide.

**API Documentation:** https://openlibrary.org/developers/api

**Search Endpoint:** `https://openlibrary.org/search.json`

**Query Parameters:**
- `title`: The book title to search for
- `limit`: Number of results to return (use 1 for top result)
- `fields`: Comma-separated list of fields to return

**Example API Response:**
```json
{
  "numFound": 1234,
  "docs": [
    {
      "title": "The Great Gatsby",
      "author_name": ["F. Scott Fitzgerald"],
      "first_publish_year": 1925,
      "subject": ["Fiction", "American literature"],
      "publisher": ["Scribner"],
      "number_of_pages_median": 180
    }
  ]
}
```

In [13]:
def fetch_book_data(title: str) -> dict:
    """
    Fetch book data from the Open Library Search API.
    
    This is a REAL API call to openlibrary.org - no API key required!
    
    Args:
        title: The book title to search for
        
    Returns:
        dict: Book information with keys:
            - title (str): Book title
            - author (str): Author name(s), comma-separated
            - year (str): First publication year
            - subjects (list): Up to 5 subject categories
            - publisher (str): Publisher name
            - pages: Number of pages (or "Unknown")
        
    Raises:
        ValueError: If book is not found or API request fails
        TimeoutError: If the API request times out
    """
    logger.info(f"Searching Open Library for: {title}")
    
    # API endpoint and parameters
    url = "https://openlibrary.org/search.json"
    params = {
        "title": title,
        "limit": 1,  # Get only the top result
        "fields": "title,author_name,first_publish_year,subject,publisher,number_of_pages_median,key"
    }
    
    try:
        # Step 1: Make the API request with timeout
        response = requests.get(url, params=params, timeout=10)
        response.raise_for_status()  # Raises exception for 4xx/5xx status codes
        
        # Step 3: Parse the JSON response
        data = response.json()
        
        # Check if any books were found
        if data.get("numFound", 0) == 0:
            raise ValueError(f"Book '{title}' not found in Open Library")
        
        # Step 4: Extract the first book and normalize the data
        book = data["docs"][0]
        
        # Build result dict with graceful handling of missing fields
        result = {
            "title": book.get("title", "Unknown Title"),
            "author": ", ".join(book.get("author_name", ["Unknown Author"])),
            "year": str(book.get("first_publish_year", "Unknown")),
            "subjects": book.get("subject", [])[:5],  # Limit to first 5 subjects
            "publisher": book.get("publisher", ["Unknown Publisher"])[0] if book.get("publisher") else "Unknown Publisher",
            "pages": book.get("number_of_pages_median", "Unknown"),
            "open_library_key": book.get("key", "")
        }
        
        logger.info(f"Found book: {result['title']} by {result['author']}")
        return result
        
    except requests.exceptions.Timeout:
        # Step 2a: Handle timeout errors
        logger.error(f"Open Library API timeout for title: {title}")
        raise TimeoutError("The Open Library API is taking too long to respond. Please try again.")
    
    except requests.exceptions.RequestException as e:
        # Step 2b: Handle other request errors
        logger.error(f"Open Library API request failed: {str(e)}")
        raise ValueError(f"Failed to fetch book data: {str(e)}")

In [14]:
# Test your fetch_book_data function
print("Testing fetch_book_data with 'The Great Gatsby'...")
try:
    book = fetch_book_data("The Great Gatsby")
    print(f"\nFound: {book['title']}")
    print(f"Author: {book['author']}")
    print(f"First Published: {book['year']}")
    print(f"Pages: {book['pages']}")
    print(f"Subjects: {', '.join(book['subjects'][:3]) if book['subjects'] else 'None'}")
    print("\nAPI helper working correctly!")
except Exception as e:
    print(f"Error: {e}")

2025-12-08 17:48:58 - book_tools - INFO - Searching Open Library for: The Great Gatsby


Testing fetch_book_data with 'The Great Gatsby'...


2025-12-08 17:48:59 - book_tools - INFO - Found book: The Great Gatsby by F. Scott Fitzgerald



Found: The Great Gatsby
Author: F. Scott Fitzgerald
First Published: 1920
Pages: 186
Subjects: Married people, fiction, American fiction (fictional works by one author), Fiction, psychological

API helper working correctly!


## Part 2: Create the Pydantic Validation Model (SOLUTION)

You are building a book information assistant. The agent should be able to look up details about books when users ask questions like "Tell me about 1984" or "Who wrote Pride and Prejudice?"

Implement a Pydantic model to validate book title queries. This protects your tool from malicious input and ensures data quality.

**Validation requirements for BookQuery:**
- Book title must be between 1 and 200 characters
- Title cannot be empty or just whitespace
- Title should only contain letters, numbers, spaces, hyphens, apostrophes, colons, and periods

In [15]:
class BookQuery(BaseModel):
    """
    Validates a book title query.
    
    Attributes:
        title (str): The book title to search for (1-200 characters)
    
    Validation rules:
        - Title must be stripped of leading/trailing whitespace
        - Title cannot be empty after stripping
        - Only allow: letters, numbers, spaces, hyphens, apostrophes, colons, periods
    """
    title: str = Field(
        ...,
        min_length=1,
        max_length=200,
        description="The book title to search for"
    )
    
    @field_validator('title')
    @classmethod
    def validate_title(cls, v):
        """
        Sanitize and validate the book title.
        
        Should:
        1. Strip whitespace from the title
        2. Raise ValueError if title is empty after stripping
        3. Raise ValueError if title contains invalid characters
           (only allow: alphanumeric, space, hyphen, apostrophe, colon, period)
        4. Return the cleaned title
        """
        # Step 1: Strip whitespace
        v = v.strip()
        
        # Step 2: Check if empty after stripping
        if not v:
            raise ValueError("Book title cannot be empty or just whitespace")
        
        # Step 3: Validate characters
        # Allow: alphanumeric, space, hyphen, apostrophe, colon, period
        allowed_chars = " -':."
        if not all(c.isalnum() or c in allowed_chars for c in v):
            raise ValueError(
                "Book title contains invalid characters. "
                "Only letters, numbers, spaces, hyphens, apostrophes, colons, and periods are allowed."
            )
        
        # Step 4: Return the cleaned title
        return v

## Part 3: Create the Book Information Tool (SOLUTION)

Implement a LangChain tool that fetches book information from the Open Library API with proper logging and error handling.

**Tool requirements:**
- Log when the tool is invoked with the book title
- Validate input using your Pydantic model
- Log success with execution duration
- Handle errors gracefully and return user-friendly messages
- Return a formatted string with book details (title, author, year, subjects)

In [16]:
@tool
def get_book_info(title: str) -> str:
    """
    Get information about a book by its title using the Open Library API.
    
    Use this tool when the user asks about a specific book, wants to know
    who wrote a book, when it was published, or what it's about.
    
    Args:
        title (str): The title of the book to look up (e.g., "1984", "Pride and Prejudice")
        
    Returns:
        str: A formatted string containing:
             - Book title and publication year
             - Author name(s)
             - Number of pages (if available)
             - Subject categories
             
    Example output format:
        "Book: 1984 (1949)
         Author: George Orwell
         Pages: 328
         Subjects: Dystopian fiction, Political fiction, Science fiction"
    """
    # Step 1: Log tool invocation
    logger.info(f"Tool invoked: get_book_info(title={title})")
    
    # Step 2: Record start time for duration tracking
    start_time = time.time()
    
    try:
        # Step 3a: Validate input using BookQuery
        query = BookQuery(title=title)
        logger.info(f"Input validation passed for title: {query.title}")
        
        # Step 3b: Fetch book data using fetch_book_data()
        book_data = fetch_book_data(query.title)
        
        # Step 3c: Format and return the result string
        subjects_str = ", ".join(book_data['subjects']) if book_data['subjects'] else "Not available"
        
        result = (
            f"Book: {book_data['title']} ({book_data['year']})\n"
            f"Author: {book_data['author']}\n"
            f"Pages: {book_data['pages']}\n"
            f"Subjects: {subjects_str}"
        )
        
        # Step 3d: Log success with duration
        duration = time.time() - start_time
        logger.info(f"Tool completed successfully in {duration:.2f}s")
        
        return result
        
    except ValueError as e:
        # Step 3e: Catch ValueError and return user-friendly error message
        duration = time.time() - start_time
        logger.error(f"Validation error after {duration:.2f}s: {str(e)}")
        return f"Error: {str(e)}"
    
    except TimeoutError as e:
        # Step 3f: Catch TimeoutError and return timeout message
        duration = time.time() - start_time
        logger.error(f"Timeout error after {duration:.2f}s: {str(e)}")
        return f"Error: {str(e)}"
    
    except Exception as e:
        # Step 3g: Catch any other Exception and return generic error message
        duration = time.time() - start_time
        logger.error(f"Unexpected error after {duration:.2f}s: {str(e)}", exc_info=True)
        return (
            f"An unexpected error occurred while fetching book data: {str(e)}. "
            "Please try again or contact support if the problem persists."
        )

## Test Your Validation Model

Run these tests to verify your Pydantic model works correctly. All tests should pass before moving on.

In [17]:
# Test valid inputs
print("Testing valid inputs...")
try:
    query = BookQuery(title="1984")
    print(f"  Valid: '{query.title}'")
except Exception as e:
    print(f"  FAILED: {e}")

try:
    query = BookQuery(title="  The Great Gatsby  ")  # Should strip whitespace
    print(f"  Valid (stripped): '{query.title}'")
except Exception as e:
    print(f"  FAILED: {e}")

try:
    query = BookQuery(title="Harry Potter: The Philosopher's Stone")  # Apostrophe and colon
    print(f"  Valid (special chars): '{query.title}'")
except Exception as e:
    print(f"  FAILED: {e}")

# Test invalid inputs
print("\nTesting invalid inputs (should fail)...")

try:
    query = BookQuery(title="   ")  # Empty after strip
    print(f"  FAILED: Should have rejected whitespace-only title")
except ValueError as e:
    print(f"  Correctly rejected whitespace: OK")

try:
    query = BookQuery(title="Book<script>alert('xss')</script>")  # Invalid chars
    print(f"  FAILED: Should have rejected special characters")
except ValueError as e:
    print(f"  Correctly rejected special chars: OK")

try:
    query = BookQuery(title="Book; DROP TABLE books;--")  # SQL injection attempt
    print(f"  FAILED: Should have rejected SQL injection")
except ValueError as e:
    print(f"  Correctly rejected SQL injection: OK")

Testing valid inputs...
  Valid: '1984'
  Valid (stripped): 'The Great Gatsby'
  Valid (special chars): 'Harry Potter: The Philosopher's Stone'

Testing invalid inputs (should fail)...
  Correctly rejected whitespace: OK
  Correctly rejected special chars: OK
  Correctly rejected SQL injection: OK


## Test Your Tool

Run these tests to verify your tool works correctly with the real Open Library API. Watch the logs to see the execution flow.

In [18]:
# Test with a classic book (real API call!)
print("Testing get_book_info with '1984'...\n")
result = get_book_info.invoke({"title": "1984"})
print(f"Result:\n{result}")

2025-12-08 17:49:01 - book_tools - INFO - Tool invoked: get_book_info(title=1984)
2025-12-08 17:49:01 - book_tools - INFO - Input validation passed for title: 1984
2025-12-08 17:49:01 - book_tools - INFO - Searching Open Library for: 1984


Testing get_book_info with '1984'...



2025-12-08 17:49:03 - book_tools - INFO - Found book: 1984 (adaptation) by Michael Dean, George Orwell
2025-12-08 17:49:03 - book_tools - INFO - Tool completed successfully in 1.22s


Result:
Book: 1984 (adaptation) (2003)
Author: Michael Dean, George Orwell
Pages: 72
Subjects: Totalitarianism, Fiction, Totalitarianism -- Fiction, Dystopias, Textbooks for foreign speakers


In [19]:
# Test with another book
print("Testing get_book_info with 'To Kill a Mockingbird'...\n")
result = get_book_info.invoke({"title": "To Kill a Mockingbird"})
print(f"Result:\n{result}")

2025-12-08 17:49:03 - book_tools - INFO - Tool invoked: get_book_info(title=To Kill a Mockingbird)
2025-12-08 17:49:03 - book_tools - INFO - Input validation passed for title: To Kill a Mockingbird
2025-12-08 17:49:03 - book_tools - INFO - Searching Open Library for: To Kill a Mockingbird


Testing get_book_info with 'To Kill a Mockingbird'...



2025-12-08 17:49:03 - book_tools - INFO - Found book: To Kill a Mockingbird by Harper Lee
2025-12-08 17:49:03 - book_tools - INFO - Tool completed successfully in 0.79s


Result:
Book: To Kill a Mockingbird (1960)
Author: Harper Lee
Pages: 322
Subjects: fiction, fiction classics, contemporary fiction, racial segregation, mob mentality


In [20]:
# Test with invalid input (should handle gracefully)
print("Testing get_book_info with invalid input...\n")
result = get_book_info.invoke({"title": "<script>bad</script>"})
print(f"Result:\n{result}")

2025-12-08 17:49:03 - book_tools - INFO - Tool invoked: get_book_info(title=<script>bad</script>)
2025-12-08 17:49:03 - book_tools - ERROR - Validation error after 0.00s: 1 validation error for BookQuery
title
  Value error, Book title contains invalid characters. Only letters, numbers, spaces, hyphens, apostrophes, colons, and periods are allowed. [type=value_error, input_value='<script>bad</script>', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/value_error


Testing get_book_info with invalid input...

Result:
Error: 1 validation error for BookQuery
title
  Value error, Book title contains invalid characters. Only letters, numbers, spaces, hyphens, apostrophes, colons, and periods are allowed. [type=value_error, input_value='<script>bad</script>', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/value_error


In [21]:
# Test with a book that doesn't exist
print("Testing get_book_info with non-existent book...\n")
result = get_book_info.invoke({"title": "xyznonexistentbook123"})
print(f"Result:\n{result}")

2025-12-08 17:49:03 - book_tools - INFO - Tool invoked: get_book_info(title=xyznonexistentbook123)
2025-12-08 17:49:03 - book_tools - INFO - Input validation passed for title: xyznonexistentbook123
2025-12-08 17:49:03 - book_tools - INFO - Searching Open Library for: xyznonexistentbook123


Testing get_book_info with non-existent book...



2025-12-08 17:49:04 - book_tools - ERROR - Validation error after 0.86s: Book 'xyznonexistentbook123' not found in Open Library


Result:
Error: Book 'xyznonexistentbook123' not found in Open Library


## Create and Test the Agent

Create an agent with your book tool and test it with natural language queries.

In [22]:
# Create the agent
model = ChatOpenAI(model="gpt-4o-mini", temperature=0.1)
book_agent = create_agent(model=model, tools=[get_book_info])

print("Book agent created successfully!")

Book agent created successfully!


In [23]:
# Test the agent with a natural language query
query = "Tell me about the book 'The Hobbit'. Who wrote it?"
print(f"Query: {query}\n")

result = book_agent.invoke({
    "messages": [{"role": "user", "content": query}]
})

print(f"Response: {result['messages'][-1].content}")

Query: Tell me about the book 'The Hobbit'. Who wrote it?



2025-12-08 17:49:07 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-12-08 17:49:07 - book_tools - INFO - Tool invoked: get_book_info(title=The Hobbit)
2025-12-08 17:49:07 - book_tools - INFO - Input validation passed for title: The Hobbit
2025-12-08 17:49:07 - book_tools - INFO - Searching Open Library for: The Hobbit
2025-12-08 17:49:09 - book_tools - INFO - Found book: The Hobbit by J.R.R. Tolkien
2025-12-08 17:49:09 - book_tools - INFO - Tool completed successfully in 1.72s
2025-12-08 17:49:11 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


Response: "The Hobbit" was written by J.R.R. Tolkien and was published in 1937. The book has 310 pages and covers subjects such as the Arkenstone, the Battle of Five Armies, invisibility, thrushes, and eagles.


In [24]:
# Test with another query
query = "What is 'Dune' about and when was it published?"
print(f"Query: {query}\n")

result = book_agent.invoke({
    "messages": [{"role": "user", "content": query}]
})

print(f"Response: {result['messages'][-1].content}")

Query: What is 'Dune' about and when was it published?



2025-12-08 17:49:12 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-12-08 17:49:12 - book_tools - INFO - Tool invoked: get_book_info(title=Dune)
2025-12-08 17:49:12 - book_tools - INFO - Input validation passed for title: Dune
2025-12-08 17:49:12 - book_tools - INFO - Searching Open Library for: Dune
2025-12-08 17:49:13 - book_tools - INFO - Found book: Dune by Frank Herbert
2025-12-08 17:49:13 - book_tools - INFO - Tool completed successfully in 1.55s
2025-12-08 17:49:17 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


Response: "Dune" is a science fiction novel written by Frank Herbert, published in 1965. The book is set in the imaginary desert planet of Arrakis, also known as Dune, and explores themes of politics, religion, and ecology. It follows the story of Paul Atreides, a young nobleman whose family becomes embroiled in a complex struggle for control of the planet and its valuable resource, the spice melange.

Here are some details about the book:
- **Author:** Frank Herbert
- **Pages:** 592
- **Subjects:** Dune (Imaginary place), Fiction, Science fiction, General fiction.
